# 08 — Export
**Goal:** Save figures at the correct format, size, and resolution for journal and conference submission.

By the end of this notebook you will:
- Know why PDF/SVG beats PNG for papers
- Use the correct `figsize` for NeurIPS, ICML, ACL, and Nature column widths
- Use `savefig` with the right parameters every time
- Verify your figure at actual print size
- Write a reproducible figure-generation script (not just a notebook)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

plt.style.use('research.mplstyle')

np.random.seed(0)

# Re-use the training curve data from earlier notebooks
epochs     = np.arange(1, 101)
train_mean = 2.5 * np.exp(-0.09 * epochs) + 0.10
train_std  = 0.06 * np.ones(100)
val_mean   = 2.5 * np.exp(-0.07 * epochs) + 0.15
val_std    = 0.09 * np.ones(100)

os.makedirs('figures', exist_ok=True)

---
## Part 1 — Vector vs. raster formats

| Format | Type | Use for |
|--------|------|--------|
| PDF | Vector | Final paper submission — scales infinitely, no pixellation |
| SVG | Vector | Web, slides, further editing in Illustrator/Inkscape |
| EPS | Vector | Some older LaTeX workflows |
| PNG | Raster | Notebooks, Slack previews, fallback |
| JPEG | Raster | Never use for plots — compression artifacts |

**Rule:** Always submit PDF to conferences. Always generate your figures programmatically (not saved from a notebook UI) so they're reproducible.

**Exercise:** Save the training curve as both PDF and PNG and compare file sizes.

In [ ]:
def make_training_figure(figsize):
    """Build a clean training curve figure. Returns fig, ax."""
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(epochs, train_mean, color='#0072B2', linewidth=1.8, label='Train')
    ax.fill_between(epochs, train_mean - train_std, train_mean + train_std, alpha=0.15, color='#0072B2')
    ax.plot(epochs, val_mean, color='#D55E00', linewidth=1.8, linestyle='--', label='Val')
    ax.fill_between(epochs, val_mean - val_std, val_mean + val_std, alpha=0.15, color='#D55E00')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    return fig, ax

# TODO: call make_training_figure((5, 3.5)) and save it as:
#   'figures/training_curve.pdf'
#   'figures/training_curve.png'
# Use fig.savefig('path', bbox_inches='tight')
# After saving, print the file size of each
# Hint: os.path.getsize('path') returns bytes


---
## Part 2 — The `savefig` parameters that matter

```python
fig.savefig(
    'figures/fig1.pdf',
    bbox_inches='tight',   # ALWAYS — prevents label clipping
    dpi=300,               # only matters for raster (PNG); ignored for PDF
    transparent=False,     # True for figures on colored slide backgrounds
    pad_inches=0.05,       # small padding around the figure
)
```

**Exercise:** Save three versions of the figure — each with a different parameter changed. Observe the differences.
1. Without `bbox_inches='tight'` (notice clipping)
2. PNG at `dpi=72` vs `dpi=300`
3. With `transparent=True` on a colored background

In [ ]:
# Experiment 1: without bbox_inches='tight'
# TODO: save with a long y-label that might get clipped
#   set y-label to 'Cross-Entropy Loss (nats)' and try saving without bbox_inches
#   then with bbox_inches='tight' — compare the two PDFs


In [ ]:
# Experiment 2: DPI comparison (raster only)
# TODO: save the figure as PNG at dpi=72, dpi=150, dpi=300
# Print file sizes for each
# Open the 72dpi version and notice the pixellation


In [ ]:
# Experiment 3: transparent background
# TODO: save with transparent=True
# Then display it on a gray background to see the effect:
#   from IPython.display import Image; Image('figures/training_transparent.png')
#   (you'll need to open it on a dark background to see the transparency)


---
## Part 3 — Column widths for major venues

Figures must fit within the paper's column width. If you set `figsize` to the correct width, the figure will be inserted at 1:1 scale in LaTeX and look exactly as designed.

| Venue | Single column | Double column / full width |
|-------|--------------|---------------------------|
| NeurIPS / ICML | 3.25 in | 6.75 in |
| ICLR | 3.25 in | 6.75 in |
| ACL (two-column) | 3.15 in | 6.52 in |
| Nature | 3.5 in | 7.2 in |
| Science | 2.25 in | 4.76 in (1.5 col) / 7.0 in (full) |

**Exercise:** Save the training curve at NeurIPS single-column width and at NeurIPS full width, then compare how they look.

In [ ]:
NEURIPS_SINGLE = (3.25, 2.5)    # width × height in inches
NEURIPS_FULL   = (6.75, 2.5)

# TODO: save figures/training_neurips_single.pdf at NEURIPS_SINGLE size
# TODO: save figures/training_neurips_full.pdf at NEURIPS_FULL size
# Open both PDFs and compare — does text stay readable at the smaller size?


---
## Part 4 — Verifying at actual print size

A critical step most people skip: open the PDF in a viewer, set zoom to 100%, and check:
- Are font sizes readable?
- Are lines visible (not too thin)?
- Are colors distinguishable?

In code, you can simulate this by rendering the figure at its exact physical size.

**Exercise:** Render a figure at NeurIPS single-column size and print the pixel dimensions at 72 DPI (screen) and 300 DPI (print).

In [ ]:
width_in, height_in = NEURIPS_SINGLE

# TODO: compute and print pixel dimensions at 72 DPI and 300 DPI
# For each DPI: pixels = inches × DPI
# print(f'At 72 DPI (screen):  {width_in * 72:.0f} × {height_in * 72:.0f} px')
# print(f'At 300 DPI (print):  {width_in * 300:.0f} × {height_in * 300:.0f} px')

# TODO: display the NeurIPS single-column figure inline in the notebook
# Does 10pt axis label text look readable?


---
## Part 5 — Reproducible figure scripts

Notebooks are great for exploring, but final paper figures should be generated by **standalone Python scripts**. This way:
- A reviewer or collaborator can reproduce every figure with `python figures/fig1.py`
- You never accidentally regenerate a figure with different data
- CI/CD can regenerate all figures automatically

**Exercise:** Write a standalone script `figures/fig1.py` that generates and saves the training curve figure.

In [ ]:
# TODO: write the contents of figures/fig1.py here, then save it
# The script should:
#   1. Import libraries and load research.mplstyle
#   2. Define / load the data
#   3. Build the figure
#   4. Call fig.savefig('figures/fig1.pdf', bbox_inches='tight')
#   5. Print 'Saved figures/fig1.pdf'

script_content = """
# TODO: fill this in
"""

with open('figures/fig1.py', 'w') as f:
    f.write(script_content)

print('Script saved to figures/fig1.py')

In [ ]:
# TODO: run the script to verify it works
# !conda run -n d2l python figures/fig1.py


---
## Part 6 — LaTeX integration tip

In your LaTeX paper, include the figure like this:

```latex
\begin{figure}[t]
  \centering
  \includegraphics[width=\columnwidth]{figures/fig1.pdf}
  \caption{Training loss curves for our model vs. baseline.}
  \label{fig:training}
\end{figure}
```

Using `width=\columnwidth` means the figure is inserted at exactly the column width — which matches your `figsize` if you sized it correctly in Part 3.

**Exercise:** Verify the mapping by computing: if LaTeX sets a figure to `\columnwidth = 3.25 in`, what `figsize` should you use so the figure is at 1:1 scale?

In [ ]:
# This is a conceptual exercise — write your answer as a comment or print statement

# TODO: if LaTeX columnwidth = 3.25 in and you want the figure height = 2.4 in,
#   what should figsize be?
# TODO: what happens if you use figsize=(6, 5) but include at \columnwidth=3.25 in?
#   (think: will font sizes look correct?)


---
## Summary

| Rule | Detail |
|------|-------|
| Format | PDF for papers, PNG for notebooks/slides |
| `bbox_inches='tight'` | Always — prevents label clipping |
| `dpi=300` | Only matters for PNG, not PDF |
| `figsize` = column width | NeurIPS/ICML: 3.25 in single, 6.75 in full |
| Verify at 100% zoom | Open PDF in viewer at 100% — check readability |
| Reproducibility | Final figures = standalone scripts, not notebook cells |

---
## You've completed the tutorial!

You now have a complete toolkit:
- A reusable style sheet (`research.mplstyle`)
- Clean foundations (OO API, spines, grid, fonts)
- Intentional color palettes (Okabe-Ito, sequential, diverging)
- Training curves with confidence bands and annotations
- Benchmark comparison charts (bar, horizontal, lollipop)
- Distribution plots (violin, KDE, ECDF, strip)
- Multi-panel figures with GridSpec and panel labels
- Reproducible, publication-ready export

**Next step:** pick a figure from your current research paper and rebuild it using what you've learned.